# Gymnasium composition

Use standard Gymnasium wrappers for task statistics and flat observations, and SyncVectorEnv to batch several independently constructed tasks.


In [ ]:
"""Compose task repetition with standard Gymnasium wrappers and vectors."""

import gymnasium as gym

from repeat_task_gym import RepeatTaskEnv


def make_task():
    return RepeatTaskEnv(
        gym.make("CartPole-v1", max_episode_steps=5), max_task_episodes=2
    )


def main():
    env = gym.wrappers.RecordEpisodeStatistics(make_task())
    try:
        env.reset(seed=0)
        env.action_space.seed(0)
        while True:
            _, _, terminated, truncated, info = env.step(env.action_space.sample())
            if terminated or truncated:
                print("Task statistics:", info["episode"])
                assert info["episode"]["l"] == 11  # Ten actions + one reset frame.
                break
    finally:
        env.close()

    flat = gym.wrappers.FlattenObservation(make_task())
    try:
        observation, _ = flat.reset(seed=0)
        assert observation in flat.observation_space
        print("Flat observation shape:", observation.shape)
    finally:
        flat.close()

    vector = gym.vector.SyncVectorEnv([make_task, make_task])
    try:
        observations, _ = vector.reset(seed=0)
        vector.action_space.seed(0)
        assert observations["episode_done"].shape == (2,)
        for _ in range(30):
            observations, rewards, terminated, truncated, _ = vector.step(
                vector.action_space.sample()
            )
        print("Vector observation shape:", observations["observation"].shape)
    finally:
        vector.close()


if __name__ == "__main__":
    main()
